# Bonsai Image — Colab Setup (serve.sh + wait)
Single cell. Starts the repo's server and waits for the backend to be ready.

In [ ]:
# setup + start serve.sh + wait for backend
import os
import subprocess
import json
import time
from pathlib import Path
import requests as _requests
ROOT = Path('.').resolve()
REPO = ROOT / 'Bonsai-Image-Demo'
STATUS_FILE = Path('/tmp/bonsai_status.json')

def run(cmd, capture=False, cwd=None):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=capture, cwd=cwd)
    if capture:
        return p.returncode, p.stdout, p.stderr
    if p.stdout:
        print(p.stdout.rstrip())
    if p.returncode != 0:
        print(f'FAILED ({p.returncode}): {cmd}')
        if p.stderr:
            print(p.stderr.rstrip())
    return p.returncode

# 1) clone
if not REPO.exists():
    run('git clone https://github.com/PrismML-Eng/Bonsai-Image-Demo.git')

# 2) setup
run(f'chmod +x {REPO}/setup.sh')
run(f'cd {REPO} && ./setup.sh')

# 3) start serve.sh
print('\nStarting serve.sh...')
serve_proc = subprocess.Popen(
    ['bash', f'{REPO}/scripts/serve.sh'],
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

def _pipe(pipe):
    for line in iter(pipe.readline, b''):
        try:
            print(line.decode('utf-8', errors='replace').rstrip())
        except Exception:
            pass
    pipe.close()

threading.Thread(target=_pipe, args=(serve_proc.stdout,), daemon=True).start()

# 4) wait for backend
ready = False
for i in range(180):
    try:
        r = _requests.get('http://127.0.0.1:8000/backends', timeout=2)
        if r.status_code == 200:
            print(f'Backend ready after {i}s')
            ready = True
            break
    except Exception:
        pass
    if i % 10 == 0 and i:
        print(f'Waiting... ({i}s)')
    time.sleep(1)

status = {
    'local_url': 'http://127.0.0.1:8000',
    'generate_endpoint': 'http://127.0.0.1:8000/generate',
    'backends_endpoint': 'http://127.0.0.1:8000/backends',
    'ready': ready,
}
STATUS_FILE.write_text(json.dumps(status, indent=2))

print('\n' + '='*60)
print('READY')
print(f'Local URL: {status["local_url"]}')
print(f'Generate:  {status["generate_endpoint"]}')
print(f'Backends:  {status["backends_endpoint"]}')
print('='*60)
